# Alternate Cooling-Flow Setup

This notebook evaluates the modified-Plummer halo variant and the analytic hot-mode accretion estimate.


In [ ]:
import sys
from pathlib import Path

def _find_repo_root():
    for start in (Path.cwd(), Path.cwd().resolve()):
        for candidate in (start, *start.parents):
            if (candidate / 'pysrc' / 'solve_ode.py').exists():
                return candidate
    raise RuntimeError('Could not find repo root containing pysrc/solve_ode.py')

repo_root = _find_repo_root()
pysrc_path = repo_root / 'pysrc'
if str(pysrc_path) not in sys.path:
    sys.path.insert(0, str(pysrc_path))

import numpy as np

import solve_ode as CF
import HaloPotential_new as Halo
import WiersmaCooling as Cool
from analytic_models import maximum_hot_mode_accretion_rate
from cosmology import DEFAULT_COSMOLOGY


In [ ]:
rho_mean = DEFAULT_COSMOLOGY.mean_matter_density_Msun_kpc3(0.0)
potential = Halo.CombinedPotential_using_modified_plummer(
    M_vir_Msun=1.0e11,
    r_vir_kpc=110.0,
    c_vir=10.0,
    M_gal_Msun=1.0e10,
    a_gal_kpc=3.0,
    b_gal_kpc=0.4,
    rho_mean_Msun_kpc3=rho_mean,
    R200_kpc=140.0,
)
cooling = Cool.Constant_Cooling(1.0e-22)
solution = CF.IntegrateFlowEquations(
    mass_flow_rate_Msun_per_yr=0.02,
    temperature_K=3.0e5,
    density_cgs=1.5e-27,
    potential=potential,
    cooling=cooling,
    direction=1,
    R_min_kpc=25.0,
    R_max_kpc=90.0,
)
mdot_hot_max = maximum_hot_mode_accretion_rate(
    vc_kms=float(potential.vc_kms(20.0)),
    R_circ_kpc=10.0,
    metallicity_solar=1.0,
)
{
    'outer_velocity_kms': float(solution.velocity_kms[-1]),
    'hot_mode_limit_Msun_per_yr': float(mdot_hot_max),
}
